<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/final/kNN_Notebook2_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# load the semi-raw data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/'

df_knn = pd.read_csv(BASE_PATH + 'networkTraffic_knn_semiraw.csv')
X = df_knn.drop(columns=['attack_cat'])
y = df_knn['attack_cat']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(X.shape, y.shape)

Mounted at /content/drive
(150243, 34) (150243,)


In [2]:
# build the per-fold pipeline: scale, bucket proto, one-hot encode.
# no clamping - tested this later on and it did not help but made the results worse so i leave it out

nominal_cols = ['proto', 'state', 'service']

fold_data_full = []
for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    proto_counts = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full.append((X_train_enc, X_test_enc, y_train, y_test))
    print(f"fold {fold_num+1}: {X_train_enc.shape}, kept protos {keep_protos}")

fold 1: (120194, 56), kept protos ['tcp', 'udp']
fold 2: (120194, 57), kept protos ['tcp', 'udp']
fold 3: (120194, 57), kept protos ['tcp', 'udp']
fold 4: (120195, 55), kept protos ['tcp', 'udp']
fold 5: (120195, 57), kept protos ['tcp', 'udp']


# Initial neighbourhood size and distance metric

In [3]:
# baseline model
f1_baseline = [f1_score(y_test, KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='euclidean').fit(X_train, y_train).predict(X_test), average='macro')
               for X_train, X_test, y_train, y_test in fold_data_full]
print(f"baseline (k=5, euclidean, uniform): {np.mean(f1_baseline):.4f} (± {np.std(f1_baseline):.4f})")

f1_weighted = [f1_score(y_test, KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean').fit(X_train, y_train).predict(X_test), average='macro')
               for X_train, X_test, y_train, y_test in fold_data_full]
print(f"distance weighted (k=5, euclidean): {np.mean(f1_weighted):.4f} (± {np.std(f1_weighted):.4f})")

baseline (k=5, euclidean, uniform): 0.3854 (± 0.0056)
distance weighted (k=5, euclidean): 0.3855 (± 0.0027)


In [4]:
# grid search over k and distance metric, single fold to keep this fast
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
metrics = ['euclidean', 'manhattan']

grid_results = []
for metric in metrics:
    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric=metric)
        model.fit(X_train_f, y_train_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results.append({'k': k, 'metric': metric, 'macro_f1': macro_f1})
        print(f"{metric:10s} k={k:3d}  {macro_f1:.4f}")

best = pd.DataFrame(grid_results).sort_values('macro_f1', ascending=False).iloc[0]
print(best)

euclidean  k=  1  0.3701
euclidean  k=  3  0.3773
euclidean  k=  5  0.3821
euclidean  k=  7  0.3807
euclidean  k=  9  0.3788
euclidean  k= 15  0.3799
euclidean  k= 21  0.3780
euclidean  k= 31  0.3703
manhattan  k=  1  0.3879
manhattan  k=  3  0.3948
manhattan  k=  5  0.4051
manhattan  k=  7  0.4088
manhattan  k=  9  0.4090
manhattan  k= 15  0.4091
manhattan  k= 21  0.4064
manhattan  k= 31  0.3944
k                  15
metric      manhattan
macro_f1     0.409056
Name: 13, dtype: object


# Feature selection

In [5]:
# feature selection - rank by mutual info, see how many features we actually need
mi_scores = mutual_info_classif(X_train_f, y_train_f, random_state=42)
mi_ranking = pd.Series(mi_scores, index=X_train_f.columns).sort_values(ascending=False)

for n in [10, 20, 30, 40]:
    top_n = mi_ranking.head(n).index.tolist()
    model = KNeighborsClassifier(n_neighbors=int(best['k']), weights='distance', metric=best['metric'])
    model.fit(X_train_f[top_n], y_train_f)
    print(f"top {n}: {f1_score(y_test_f, model.predict(X_test_f[top_n]), average='macro'):.4f}")

# top 10 is best - confirm properly across all folds, recomputing MI each time
# so nothing leaks from test fold into feature ranking

fold_data_selected = []
fold_selected_features = []
for X_train, X_test, y_train, y_test in fold_data_full:
    mi = mutual_info_classif(X_train, y_train, random_state=42)
    ranking = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    top10 = ranking.head(10).index.tolist()
    fold_selected_features.append(top10)
    fold_data_selected.append((X_train[top10], X_test[top10], y_train, y_test))

f1_mi = [f1_score(y_test, KNeighborsClassifier(n_neighbors=int(best['k']), weights='distance', metric=best['metric']).fit(X_train, y_train).predict(X_test), average='macro')
         for X_train, X_test, y_train, y_test in fold_data_selected]
print(f"confirmed top-10: {np.mean(f1_mi):.4f} (± {np.std(f1_mi):.4f})")
print(fold_selected_features)

top 10: 0.5409
top 20: 0.4702
top 30: 0.4256
top 40: 0.4079
confirmed top-10: 0.5377 (± 0.0079)
[['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dinpkt', 'dload', 'dpkts'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dpkts', 'dinpkt'], ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']]


In [6]:
mi_ranking.head(10)

,0
sbytes,0.694908
dbytes,0.572401
smean,0.464288
dmean,0.438621
sttl,0.421160
ct_state_ttl,0.350093
dttl,0.344732
dinpkt,0.324898
dload,0.323182
dpkts,0.321638


# Resampling strategy

In [7]:
# check whether the connection-count cluster reduction (7 features - 2) actually matters, by rebuilding the pipeline on a version that kept all 7
df_7c = pd.read_csv(BASE_PATH + 'networkTraffic_knn_semiraw_7cluster.csv')
X_7c = df_7c.drop(columns=['attack_cat'])
y_7c = df_7c['attack_cat']
skf_7c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_data_full_7c = []
for fold_num, (train_idx, test_idx) in enumerate(skf_7c.split(X_7c, y_7c)):
    X_train, X_test = X_7c.iloc[train_idx].copy(), X_7c.iloc[test_idx].copy()
    y_train, y_test = y_7c.iloc[train_idx], y_7c.iloc[test_idx]

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    proto_counts = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full_7c.append((X_train_enc, X_test_enc, y_train, y_test))

fold_data_selected_7c = []
for X_train, X_test, y_train, y_test in fold_data_full_7c:
    mi = mutual_info_classif(X_train, y_train, random_state=42)
    ranking = pd.Series(mi, index=X_train.columns).sort_values(ascending=False)
    top10 = ranking.head(10).index.tolist()
    fold_data_selected_7c.append((X_train[top10], X_test[top10], y_train, y_test))

f1_7c = [f1_score(y_test, KNeighborsClassifier(n_neighbors=int(best['k']), weights='distance', metric=best['metric']).fit(X_train, y_train).predict(X_test), average='macro')
         for X_train, X_test, y_train, y_test in fold_data_selected_7c]

print(f"2-feature cluster version: {np.mean(f1_mi):.4f}")
print(f"7-feature cluster version: {np.mean(f1_7c):.4f} (± {np.std(f1_7c):.4f})")
t_stat, p_val = stats.ttest_rel(f1_mi, f1_7c)
print(f"t={t_stat:.4f}, p={p_val:.4f}")
# not significant and neither version's top-10 actually includes any ofthe cluster features anyway - sticking with the 2-feature version

2-feature cluster version: 0.5377
7-feature cluster version: 0.5287 (± 0.0090)
t=1.8400, p=0.1396


In [8]:
# k needs re-checking once we know what resampling we're using, since resampling changes the density of the training data
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_selected[0]
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train_f, y_train_f)

for k in [3, 5, 7, 9, 15, 21]:
    model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='manhattan')
    model.fit(Xs, ys)
    print(f"k={k}: {f1_score(y_test_f, model.predict(X_test_f), average='macro'):.4f}")

FINAL_K = 7  # confirmed winner

k=3: 0.5302
k=5: 0.5349
k=7: 0.5405
k=9: 0.5282
k=15: 0.5352
k=21: 0.5405


In [9]:
# resampling comparison - the main one, at the confirmed k
f1_flat, f1_smote, f1_ros, f1_tomek, f1_smote_tomek = [], [], [], [], []

for X_train, X_test, y_train, y_test in fold_data_selected:
    target_flat = {cls: max(count, 5000) for cls, count in y_train.value_counts().items()}
    Xs_i, ys_i = SMOTE(sampling_strategy=target_flat, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_flat.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan').fit(Xs_i, ys_i).predict(X_test), average='macro'))

    target_capped_i = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xs_i, ys_i = SMOTE(sampling_strategy=target_capped_i, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_smote.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan').fit(Xs_i, ys_i).predict(X_test), average='macro'))

    Xr_i, yr_i = RandomOverSampler(sampling_strategy=target_capped_i, random_state=42).fit_resample(X_train, y_train)
    f1_ros.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan').fit(Xr_i, yr_i).predict(X_test), average='macro'))

    Xt_i, yt_i = TomekLinks().fit_resample(X_train, y_train)
    f1_tomek.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan').fit(Xt_i, yt_i).predict(X_test), average='macro'))

    smote_st_i = SMOTE(sampling_strategy=target_capped_i, random_state=42, k_neighbors=5)
    Xst_i, yst_i = SMOTETomek(smote=smote_st_i, random_state=42).fit_resample(X_train, y_train)
    f1_smote_tomek.append(f1_score(y_test, KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan').fit(Xst_i, yst_i).predict(X_test), average='macro'))

print(f"SMOTE flat:        {np.mean(f1_flat):.4f} (± {np.std(f1_flat):.4f})")
print(f"SMOTE 5x capped:   {np.mean(f1_smote):.4f} (± {np.std(f1_smote):.4f})")
print(f"random oversample: {np.mean(f1_ros):.4f} (± {np.std(f1_ros):.4f})")
print(f"tomek only:        {np.mean(f1_tomek):.4f} (± {np.std(f1_tomek):.4f})")
print(f"SMOTE + tomek:     {np.mean(f1_smote_tomek):.4f} (± {np.std(f1_smote_tomek):.4f})")

t_stat, p_val = stats.ttest_rel(f1_smote_tomek, f1_smote)
print(f"SMOTE+tomek vs SMOTE: t={t_stat:.4f}, p={p_val:.4f}")

SMOTE flat:        0.5288 (± 0.0078)
SMOTE 5x capped:   0.5382 (± 0.0100)
random oversample: 0.5288 (± 0.0053)
tomek only:        0.5391 (± 0.0065)
SMOTE + tomek:     0.5426 (± 0.0089)
SMOTE+tomek vs SMOTE: t=2.7932, p=0.0492


# Distance-weighting function

In [10]:
# is SMOTE's own k_neighbors setting worth changing from the default
for smote_k in [3, 5, 7, 10]:
    f1s = []
    for X_train, X_test, y_train, y_test in fold_data_selected:
        target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
        smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=smote_k)
        Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
        m = KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan')
        m.fit(Xst, yst)
        f1s.append(f1_score(y_test, m.predict(X_test), average='macro'))
    print(f"smote k_neighbors={smote_k}: {np.mean(f1s):.4f} (± {np.std(f1s):.4f})")
# 5 (the default) is still best, no change needed

smote k_neighbors=3: 0.5366 (± 0.0133)
smote k_neighbors=5: 0.5426 (± 0.0089)
smote k_neighbors=7: 0.5406 (± 0.0066)
smote k_neighbors=10: 0.5376 (± 0.0082)


# Final confirmation of distance metric and feature scaling

In [11]:
# distance weighting - is 1/d^2 better than the standard 1/d
def inverse_squared_weights(distances):
    with np.errstate(divide='ignore'):
        weights = 1.0 / (distances ** 2)
    inf_mask = np.isinf(weights)
    if np.any(inf_mask):
        for i, row_mask in enumerate(inf_mask):
            if row_mask.any():
                weights[i] = row_mask.astype(float)
    return weights

model_std = KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan')
model_std.fit(Xs, ys)
print(f"1/d:  {f1_score(y_test_f, model_std.predict(X_test_f), average='macro'):.4f}")

model_sq = KNeighborsClassifier(n_neighbors=FINAL_K, weights=inverse_squared_weights, metric='manhattan')
model_sq.fit(Xs, ys)
print(f"1/d^2: {f1_score(y_test_f, model_sq.predict(X_test_f), average='macro'):.4f}")
# standard 1/d wins, keeping that

1/d:  0.5405
1/d^2: 0.5384


In [12]:
# one more check - manhattan vs euclidean, now that resampling is confirmed
target_capped = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train_f, y_train_f)

for metric in ['manhattan', 'euclidean']:
    m = KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric=metric)
    m.fit(Xst, yst)
    print(f"{metric}: {f1_score(y_test_f, m.predict(X_test_f), average='macro'):.4f}")

manhattan: 0.5415
euclidean: 0.5352


In [13]:
# scaling - min-max vs z-score, at the final configuration
model_mm = KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan')
model_mm.fit(Xst, yst)
print(f"min-max: {f1_score(y_test_f, model_mm.predict(X_test_f), average='macro'):.4f}")

scaler_z = StandardScaler()
Xst_z = scaler_z.fit_transform(Xst)
Xtest_z = scaler_z.transform(X_test_f)
model_z = KNeighborsClassifier(n_neighbors=FINAL_K, weights='distance', metric='manhattan')
model_z.fit(Xst_z, yst)
print(f"z-score: {f1_score(y_test_f, model_z.predict(Xtest_z), average='macro'):.4f}")

min-max: 0.5415
z-score: 0.5369


# Final model

In [14]:
# final model - full 5 fold run
final_f1, final_acc, final_weighted = [], [], []
all_y_test, all_y_pred = [], []

for X_train, X_test, y_train, y_test in fold_data_selected:
    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)

    model = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    model.fit(Xst, yst)
    y_pred = model.predict(X_test)

    final_f1.append(f1_score(y_test, y_pred, average='macro'))
    final_acc.append(accuracy_score(y_test, y_pred))
    final_weighted.append(f1_score(y_test, y_pred, average='weighted'))
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"macro-F1: {np.mean(final_f1):.4f} (± {np.std(final_f1):.4f})")
print(f"accuracy: {np.mean(final_acc):.4f} (± {np.std(final_acc):.4f})")
print(f"weighted-F1: {np.mean(final_weighted):.4f} (± {np.std(final_weighted):.4f})")
print()
print(classification_report(all_y_test, all_y_pred, digits=3))

macro-F1: 0.5426 (± 0.0089)
accuracy: 0.7673 (± 0.0024)
weighted-F1: 0.7786 (± 0.0017)

              precision    recall  f1-score   support

           0      0.926     0.838     0.880     83358
           1      0.712     0.744     0.727      8609
           2      0.187     0.622     0.287      1503
           3      0.349     0.405     0.375      5019
           4      0.810     0.772     0.791     26927
           5      0.141     0.067     0.090      1577
           6      0.525     0.671     0.589     19470
           7      0.559     0.669     0.609       169
           8      0.379     0.424     0.400      1426
           9      0.732     0.628     0.676      2185

    accuracy                          0.767    150243
   macro avg      0.532     0.584     0.542    150243
weighted avg      0.798     0.767     0.779    150243



In [15]:
# save per-fold macro-F1 so it can be compared against the tree's results
import json
with open('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/knn_final_f1.json', 'w') as f:
    json.dump(final_f1, f)
print("Saved:", final_f1)

Saved: [0.541480618526235, 0.5278417525872214, 0.5542537303755382, 0.5488823402400633, 0.5403736958837997]


# Extra checks for outlier handling, feature scaling and different circumstances

In [16]:
# Clamp vs unclamped, at the final configuration
# (k=7, Manhattan, distance-weighted, SMOTE+Tomek, top-10 features)

def clamp_fold(X_train, X_test, cols, lower_q=0.01, upper_q=0.99):
    X_train_c, X_test_c = X_train.copy(), X_test.copy()
    for col in cols:
        lo, hi = X_train[col].quantile(lower_q), X_train[col].quantile(upper_q)
        X_train_c[col] = X_train[col].clip(lo, hi)
        X_test_c[col] = X_test[col].clip(lo, hi)
    return X_train_c, X_test_c

f1_unclamped, f1_clamped = [], []

for X_train, X_test, y_train, y_test in fold_data_selected:
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    # unclamped (matches current final pipeline)
    target = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_st = SMOTE(sampling_strategy=target, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
    m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    m.fit(Xst, yst)
    f1_unclamped.append(f1_score(y_test, m.predict(X_test), average='macro'))

    # clamped version — same pipeline, clamp applied before resampling
    X_train_c, X_test_c = clamp_fold(X_train, X_test, numeric_cols)
    target_c = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_c = SMOTE(sampling_strategy=target_c, random_state=42, k_neighbors=5)
    Xst_c, yst_c = SMOTETomek(smote=smote_c, random_state=42).fit_resample(X_train_c, y_train)
    m_c = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    m_c.fit(Xst_c, yst_c)
    f1_clamped.append(f1_score(y_test, m_c.predict(X_test_c), average='macro'))

print(f"Unclamped: {np.mean(f1_unclamped):.4f} (± {np.std(f1_unclamped):.4f})")
print(f"Clamped:   {np.mean(f1_clamped):.4f} (± {np.std(f1_clamped):.4f})")

t_stat, p_val = stats.ttest_rel(f1_unclamped, f1_clamped)
print(f"Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

Unclamped: 0.5426 (± 0.0089)
Clamped:   0.5378 (± 0.0100)
Paired t-test: t=2.3769, p=0.0762


In [17]:
# Log-transform test, applied before scaling (unlike the earlier clamp test, which worked on already-scaled fold_data_selected)

final_features = ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl',
                   'ct_state_ttl', 'dttl', 'dload', 'dinpkt', 'dpkts']
skewed_cols = ['sbytes', 'dbytes', 'smean', 'dmean', 'dload', 'dinpkt', 'dpkts']

f1_unclamped, f1_logtransform = [], []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train_raw = X.iloc[train_idx][final_features].copy()
    X_test_raw = X.iloc[test_idx][final_features].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # unclamped, no log transform (current final pipeline)
    scaler = MinMaxScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=final_features)
    X_test_s = pd.DataFrame(scaler.transform(X_test_raw), columns=final_features)

    target = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_st = SMOTE(sampling_strategy=target, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train_s, y_train)
    m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    m.fit(Xst, yst)
    f1_unclamped.append(f1_score(y_test, m.predict(X_test_s), average='macro'))

    # log1p on the skewed subset, applied before scaling
    X_train_log = X_train_raw.copy()
    X_test_log = X_test_raw.copy()
    for col in skewed_cols:
        X_train_log[col] = np.log1p(X_train_log[col])
        X_test_log[col] = np.log1p(X_test_log[col])

    scaler_log = MinMaxScaler()
    X_train_log_s = pd.DataFrame(scaler_log.fit_transform(X_train_log), columns=final_features)
    X_test_log_s = pd.DataFrame(scaler_log.transform(X_test_log), columns=final_features)

    target_log = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    smote_log = SMOTE(sampling_strategy=target_log, random_state=42, k_neighbors=5)
    Xst_log, yst_log = SMOTETomek(smote=smote_log, random_state=42).fit_resample(X_train_log_s, y_train)
    m_log = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
    m_log.fit(Xst_log, yst_log)
    f1_logtransform.append(f1_score(y_test, m_log.predict(X_test_log_s), average='macro'))

print(f"Unclamped, no log (current final): {np.mean(f1_unclamped):.4f} (± {np.std(f1_unclamped):.4f})")
print(f"Log-transformed (skewed subset):   {np.mean(f1_logtransform):.4f} (± {np.std(f1_logtransform):.4f})")

t_stat, p_val = stats.ttest_rel(f1_unclamped, f1_logtransform)
print(f"Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

Unclamped, no log (current final): 0.5426 (± 0.0089)
Log-transformed (skewed subset):   0.5248 (± 0.0083)
Paired t-test: t=4.2071, p=0.0136


In [18]:
# SMOTE oversampling ratio sensitivity, on the final configuration
# (unclamped, k=7, Manhattan, SMOTE+Tomek)

for ratio in [2, 3, 5, 10]:
    f1s = []
    for X_train, X_test, y_train, y_test in fold_data_selected:
        target = {cls: min(count * ratio, y_train.value_counts().max())
                   for cls, count in y_train.value_counts().items()}
        smote_st = SMOTE(sampling_strategy=target, random_state=42, k_neighbors=5)
        Xst, yst = SMOTETomek(smote=smote_st, random_state=42).fit_resample(X_train, y_train)
        m = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan')
        m.fit(Xst, yst)
        f1s.append(f1_score(y_test, m.predict(X_test), average='macro'))
    print(f"ratio={ratio}x: {np.mean(f1s):.4f} (± {np.std(f1s):.4f})")

ratio=2x: 0.5402 (± 0.0106)
ratio=3x: 0.5369 (± 0.0130)
ratio=5x: 0.5426 (± 0.0089)
ratio=10x: 0.5344 (± 0.0092)
